# 02 — Silver: features das transações

Espelha `RETREINO/00_struct_table.py`: instala `haversine`, registra `dist_haversine` e deriva as features em Spark SQL. Adapta a origem para `desloc.bronze.transacoes`, grava a tabela permanente `desloc.silver.transacoes` e chama a distância calculada de `distancia_lalo`.

A regra de período segue sua definição anterior: 1 = 07h–11h; 2 = 12h–17h; 3 = demais horários. A data/hora completa não é gravada na Silver.

In [1]:
!pip install haversine

In [2]:
%restart_python

In [3]:
from haversine import haversine
from pyspark.sql.types import FloatType

def dist_haversine(lat_o, lon_o, lat_d, lon_d):
    return haversine((lat_o, lon_o), (lat_d, lon_d))

spark.udf.register("dist_haversine", dist_haversine, FloatType())

<function __main__.dist_haversine(lat_o, lon_o, lat_d, lon_d)>

In [4]:
spark.sql("""
    CREATE OR REPLACE TABLE desloc.silver.transacoes AS
    WITH tbl_features AS (
    SELECT
        day(to_timestamp(datahora_entregue, 'dd/MM/yyyy HH:mm')) as dia,
        dayofweek(to_timestamp(datahora_entregue, 'dd/MM/yyyy HH:mm')) as dia_semana,
        month(to_timestamp(datahora_entregue, 'dd/MM/yyyy HH:mm')) as mes,
        hour(to_timestamp(datahora_entregue, 'dd/MM/yyyy HH:mm')) as hora,
        duracao,
        distancia,
        vlr_pago,
        latitude_origem,
        longitude_origem,
        latitude_destino,
        longitude_destino
    FROM desloc.bronze.transacoes
    )
    SELECT
    dia,
    dia_semana,
    mes,
    hora,
    case when hora < 12 and hora > 6 then 1
        when hora >= 12 and hora < 18 then 2
        else 3 end as periodo,
    duracao,
    vlr_pago,
    latitude_origem,
    longitude_origem,
    latitude_destino,
    longitude_destino,
    dist_haversine(latitude_origem, longitude_origem, latitude_destino, longitude_destino) as distancia_lalo
    FROM tbl_features
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

## Conferência
A consulta abaixo apenas mostra uma amostra da tabela permanente criada acima.

In [5]:
display(spark.sql("SELECT * FROM desloc.silver.transacoes LIMIT 5"))

dia,dia_semana,mes,hora,periodo,duracao,vlr_pago,latitude_origem,longitude_origem,latitude_destino,longitude_destino,distancia_lalo
3,2,11,7,1,12,23.97,-23.513742609587695,-46.62327810242303,-23.483853341186528,-46.648585835169115,4.2078524
23,5,10,9,1,11,24.71,-23.51207921873963,-46.62739562527016,-23.474032512968865,-46.63578672625711,4.3162794
22,4,10,11,1,9,23.01,-23.574913898123835,-46.60181176389887,-23.611927332177384,-46.592856334210495,4.215667
18,7,10,18,3,11,21.2,-23.581033986174024,-46.63696302378459,-23.598154937550955,-46.66461159876348,3.4003665
13,2,10,11,1,11,21.74,-23.581544813632622,-46.643830193209304,-23.56791182574625,-46.611816490410796,3.5976384


## Limite para previsão antecipada
`dia`, `dia_semana`, `mes` e `hora` derivam do horário de entrega. Para cotar o preço antes da corrida, é necessária a data/hora disponível na solicitação ou coleta.